# Minimal Burgers Discovery Story

This notebook is a deliberately bare-bones diagnostic. It uses the existing `burg_gen` dataset generator, the current `SirenMLP` surrogate, and the project's Burgers reference-derivative helper, but keeps the actual fitting, autodiff, least-squares identification, and minimal SymNet logic visible in notebook cells.

Questions answered in order:

1. What does the clean Burgers rollout look like?
2. Can the current SIREN fit the solution from data alone?
3. Do the fitted surrogate derivatives match the reference derivatives well enough?
4. Can direct least squares on surrogate derivatives recover Burgers?
5. What is the smallest visible SymNet-style product model that can express Burgers?
6. Does that architecture reproduce Burgers exactly when weights are hand-set?
7. If we start exactly at Burgers and train only SymNet, does it stay there or drift?
8. Optionally, what changes if SIREN and SymNet are trained jointly?

In [ ]:
from __future__ import annotations

import copy
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from Datasets.data.processed.burg_gen.burg_gen import solve_burgers
from prog.mlps import SirenMLP
from utils.derivative_utils import build_burgers_reference_derivative_grids, compute_error_metrics

plt.style.use("seaborn-v0_8-whitegrid")

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
print(f"seed = {SEED}")

In [ ]:
def plot_time_slices(x, true_grid, pred_grid=None, time_values=None, title_prefix="u", slice_indices=None):
    if slice_indices is None:
        slice_indices = np.linspace(0, len(time_values) - 1, 5, dtype=int)
    ncols = len(slice_indices)
    fig, axes = plt.subplots(1, ncols, figsize=(4 * ncols, 3.5), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, idx in zip(axes, slice_indices):
        ax.plot(x, true_grid[idx], label=f"true {title_prefix}", linewidth=2)
        if pred_grid is not None:
            ax.plot(x, pred_grid[idx], "--", label=f"pred {title_prefix}", linewidth=2)
        ax.set_title(f"t = {time_values[idx]:.3f}")
        ax.set_xlabel("x")
    axes[0].set_ylabel(title_prefix)
    axes[0].legend(loc="best")
    plt.tight_layout()
    plt.show()


def evaluate_model_and_derivatives_in_chunks(model, t_flat, x_flat, grid_shape, chunk_size=4096, device=device):
    u_parts = []
    ut_parts = []
    ux_parts = []
    uxx_parts = []

    model.eval()
    for start in range(0, len(t_flat), chunk_size):
        stop = min(start + chunk_size, len(t_flat))

        t_chunk = torch.tensor(t_flat[start:stop], dtype=torch.float32, device=device).reshape(-1, 1)
        x_chunk = torch.tensor(x_flat[start:stop], dtype=torch.float32, device=device).reshape(-1, 1)
        t_chunk.requires_grad_(True)
        x_chunk.requires_grad_(True)

        u_chunk = model(t_chunk, x_chunk)
        ut_chunk = torch.autograd.grad(
            u_chunk,
            t_chunk,
            grad_outputs=torch.ones_like(u_chunk),
            create_graph=False,
            retain_graph=True,
        )[0]
        ux_chunk = torch.autograd.grad(
            u_chunk,
            x_chunk,
            grad_outputs=torch.ones_like(u_chunk),
            create_graph=True,
            retain_graph=True,
        )[0]
        uxx_chunk = torch.autograd.grad(
            ux_chunk,
            x_chunk,
            grad_outputs=torch.ones_like(ux_chunk),
            create_graph=False,
            retain_graph=False,
        )[0]

        u_parts.append(u_chunk.detach().cpu().numpy())
        ut_parts.append(ut_chunk.detach().cpu().numpy())
        ux_parts.append(ux_chunk.detach().cpu().numpy())
        uxx_parts.append(uxx_chunk.detach().cpu().numpy())

    u = np.concatenate(u_parts, axis=0).reshape(grid_shape)
    ut = np.concatenate(ut_parts, axis=0).reshape(grid_shape)
    ux = np.concatenate(ux_parts, axis=0).reshape(grid_shape)
    uxx = np.concatenate(uxx_parts, axis=0).reshape(grid_shape)
    return {"u": u, "u_t": ut, "u_x": ux, "u_xx": uxx}


def as_metric_row(name, pred, ref):
    row = compute_error_metrics(pred, ref)
    row["quantity"] = name
    return row


def product_term_dict(model):
    left = model.left.weight.detach().cpu().numpy().reshape(-1)
    right = model.right.weight.detach().cpu().numpy().reshape(-1)
    linear = model.linear.weight.detach().cpu().numpy().reshape(-1)
    alpha = float(model.product_readout.weight.detach().cpu().numpy().reshape(-1)[0])
    names = ["u", "u_x", "u_xx"]

    coeffs = {
        "u": float(linear[0]),
        "u_x": float(linear[1]),
        "u_xx": float(linear[2]),
        "u^2": float(alpha * left[0] * right[0]),
        "u*u_x": float(alpha * (left[0] * right[1] + left[1] * right[0])),
        "u*u_xx": float(alpha * (left[0] * right[2] + left[2] * right[0])),
        "u_x^2": float(alpha * left[1] * right[1]),
        "u_x*u_xx": float(alpha * (left[1] * right[2] + left[2] * right[1])),
        "u_xx^2": float(alpha * left[2] * right[2]),
    }
    return coeffs


def pretty_equation(coeffs):
    ordered = ["u", "u_x", "u_xx", "u^2", "u*u_x", "u*u_xx", "u_x^2", "u_x*u_xx", "u_xx^2"]
    pieces = [f"({coeffs[name]:+.6f})*{name}" for name in ordered if abs(coeffs[name]) > 1e-12]
    return "u_t_hat = " + (" + ".join(pieces) if pieces else "0")


def initialize_symnet_to_burgers(model, diffusion=0.02):
    with torch.no_grad():
        model.left.weight.zero_()
        model.right.weight.zero_()
        model.linear.weight.zero_()
        model.product_readout.weight.zero_()

        model.left.weight[0, 0] = 1.0
        model.right.weight[0, 1] = 1.0
        model.product_readout.weight[0, 0] = -1.0
        model.linear.weight[0, 2] = diffusion


## 1. Load and Inspect the Burgers Data

Question answered here: what clean solution are we trying to fit, and what is the true PDE behind it?

In [ ]:
NU_TRUE = 0.02

x_grid, _u_final, _t_end, (t_grid, u_grid) = solve_burgers(seed=SEED, return_history=True)
x_grid = np.asarray(x_grid, dtype=np.float32)
t_grid = np.asarray(t_grid, dtype=np.float32)
u_grid = np.asarray(u_grid, dtype=np.float32)

print(f"x shape: {x_grid.shape}")
print(f"t shape: {t_grid.shape}")
print(f"u shape: {u_grid.shape}")
print("true PDE: u_t = -u*u_x + 0.02*u_xx")

slice_indices = np.linspace(0, len(t_grid) - 1, 5, dtype=int)
slice_times = t_grid[slice_indices]
print("representative slice indices:", slice_indices)
print("representative slice times:", np.round(slice_times, 4))

In [ ]:
plot_time_slices(x_grid, u_grid, pred_grid=None, time_values=t_grid, title_prefix="u", slice_indices=slice_indices)

## 2. Fit the Neural Surrogate to Data Only

Question answered here: can the current SIREN represent the Burgers rollout when trained only on solution values, with no PDE loss or sparsity machinery?

In [ ]:
T_mesh = np.repeat(t_grid[:, None], len(x_grid), axis=1)
X_mesh = np.repeat(x_grid[None, :], len(t_grid), axis=0)

t_flat = T_mesh.reshape(-1).astype(np.float32)
x_flat = X_mesh.reshape(-1).astype(np.float32)
u_flat = u_grid.reshape(-1).astype(np.float32)

t_train = torch.tensor(t_flat, dtype=torch.float32, device=device).reshape(-1, 1)
x_train = torch.tensor(x_flat, dtype=torch.float32, device=device).reshape(-1, 1)
u_train = torch.tensor(u_flat, dtype=torch.float32, device=device).reshape(-1, 1)

print("flattened training tensors:")
print("t_train", tuple(t_train.shape))
print("x_train", tuple(x_train.shape))
print("u_train", tuple(u_train.shape))

siren = SirenMLP(
    hidden_size=64,
    hidden_layers=3,
    first_omega_0=20.0,
    hidden_omega_0=1.0,
).to(device)

adam_steps = 2500
lbfgs_steps = 200
batch_size = 4096
log_every = 250

adam = torch.optim.Adam(siren.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
data_history = []

for step in range(1, adam_steps + 1):
    idx = torch.randint(0, len(t_train), (batch_size,), device=device)
    t_batch = t_train[idx]
    x_batch = x_train[idx]
    u_batch = u_train[idx]

    pred_batch = siren(t_batch, x_batch)
    loss = loss_fn(pred_batch, u_batch)

    adam.zero_grad()
    loss.backward()
    adam.step()

    if step % log_every == 0 or step == 1:
        with torch.no_grad():
            full_loss = loss_fn(siren(t_train, x_train), u_train).item()
        data_history.append({"stage": "adam", "step": step, "loss": full_loss})
        print(f"adam step {step:4d} | full data MSE = {full_loss:.6e}")

lbfgs = torch.optim.LBFGS(siren.parameters(), lr=1.0, max_iter=lbfgs_steps, history_size=50, line_search_fn="strong_wolfe")

def lbfgs_closure():
    lbfgs.zero_grad()
    pred = siren(t_train, x_train)
    loss = loss_fn(pred, u_train)
    loss.backward()
    return loss

lbfgs_loss = float(lbfgs.step(lbfgs_closure))
with torch.no_grad():
    final_data_mse = loss_fn(siren(t_train, x_train), u_train).item()
print(f"lbfgs start loss = {lbfgs_loss:.6e}")
print(f"final full data MSE = {final_data_mse:.6e}")

In [ ]:
with torch.no_grad():
    u_pred_grid = siren(t_train, x_train).detach().cpu().numpy().reshape(u_grid.shape)

solution_metrics = pd.DataFrame([
    as_metric_row("u", u_pred_grid, u_grid),
]).set_index("quantity")
display(solution_metrics)

plot_time_slices(x_grid, u_grid, pred_grid=u_pred_grid, time_values=t_grid, title_prefix="u", slice_indices=slice_indices)

## 3. Inspect Derivatives Produced by the Fitted Surrogate

Question answered here: after fitting only to solution values, do the surrogate's autodiff derivatives `u_t`, `u_x`, and `u_xx` match the clean Burgers reference derivatives used elsewhere in the project?

In [ ]:
reference = build_burgers_reference_derivative_grids(
    u_grid=u_grid,
    x_grid=x_grid,
    nu=NU_TRUE,
)

surrogate_fields = evaluate_model_and_derivatives_in_chunks(
    siren,
    t_flat=t_flat,
    x_flat=x_flat,
    grid_shape=u_grid.shape,
    chunk_size=4096,
    device=device,
)

derivative_metrics = pd.DataFrame([
    as_metric_row("u_t", surrogate_fields["u_t"], reference["u_t"]["physical"]),
    as_metric_row("u_x", surrogate_fields["u_x"], reference["u_x"]["physical"]),
    as_metric_row("u_xx", surrogate_fields["u_xx"], reference["u_xx"]["physical"]),
]).set_index("quantity")
display(derivative_metrics)

In [ ]:
for quantity, ref_key in [("u_t", "u_t"), ("u_x", "u_x"), ("u_xx", "u_xx")]:
    plot_time_slices(
        x_grid,
        reference[ref_key]["physical"],
        pred_grid=surrogate_fields[quantity],
        time_values=t_grid,
        title_prefix=quantity,
        slice_indices=slice_indices,
    )

## 4. Recover Burgers with Direct Least Squares

Question answered here: if we freeze the surrogate and use only the Burgers basis `[u*u_x, u_xx]`, do the surrogate derivatives contain enough information for ordinary least squares to recover `u_t = -u*u_x + 0.02*u_xx`?

In [ ]:
theta = np.column_stack([
    (surrogate_fields["u"] * surrogate_fields["u_x"]).reshape(-1),
    surrogate_fields["u_xx"].reshape(-1),
])
y = surrogate_fields["u_t"].reshape(-1)

coeffs_ls, residuals_ls, rank_ls, singular_vals_ls = np.linalg.lstsq(theta, y, rcond=None)
ls_prediction = (theta @ coeffs_ls).reshape(u_grid.shape)
ls_residual_norm = float(np.linalg.norm(y - theta @ coeffs_ls))

print(f"rank(theta) = {rank_ls}")
print(f"singular values = {singular_vals_ls}")
print(f"least-squares residual L2 norm = {ls_residual_norm:.6e}")
print()
print(f"Recovered PDE: u_t = ({coeffs_ls[0]:+.6f}) * u*u_x + ({coeffs_ls[1]:+.6f}) * u_xx")
print(f"Compare c1 against -1.00  -> error = {coeffs_ls[0] + 1.0:+.6e}")
print(f"Compare c2 against +0.02 -> error = {coeffs_ls[1] - 0.02:+.6e}")

display(pd.DataFrame([
    {"coefficient": "u*u_x", "recovered": coeffs_ls[0], "truth": -1.0, "error": coeffs_ls[0] + 1.0},
    {"coefficient": "u_xx", "recovered": coeffs_ls[1], "truth": 0.02, "error": coeffs_ls[1] - 0.02},
]).set_index("coefficient"))

display(pd.DataFrame([
    as_metric_row("LS RHS vs surrogate u_t", ls_prediction, surrogate_fields["u_t"]),
]).set_index("quantity"))

## 5. Define a Minimal SymNet/EQL Model Directly in the Notebook

Question answered here: what is the smallest visible one-product architecture that can represent the Burgers right-hand side from inputs `[u, u_x, u_xx]`?

In [ ]:
class MinimalSymNet(nn.Module):
    """
    Three primitive inputs: [u, u_x, u_xx]
    One product channel: (left(features)) * (right(features))
    Final readout: linear primitive term + weighted product term
    """

    def __init__(self):
        super().__init__()
        self.left = nn.Linear(3, 1, bias=False)
        self.right = nn.Linear(3, 1, bias=False)
        self.linear = nn.Linear(3, 1, bias=False)
        self.product_readout = nn.Linear(1, 1, bias=False)

    def forward(self, features):
        z_left = self.left(features)
        z_right = self.right(features)
        product = z_left * z_right
        linear_part = self.linear(features)
        return linear_part + self.product_readout(product)


symnet_demo = MinimalSymNet()
print(symnet_demo)
print()
print("forward map:")
print("z_left  = left([u, u_x, u_xx])")
print("z_right = right([u, u_x, u_xx])")
print("product = z_left * z_right")
print("output  = linear([u, u_x, u_xx]) + product_readout(product)")

## 6. Hand-Set SymNet Weights to Burgers

Question answered here: without any training, can this minimal product architecture compute `-u*u_x + 0.02*u_xx` exactly when we assign the weights by hand?

In [ ]:
symnet_exact = MinimalSymNet().to(device)
initialize_symnet_to_burgers(symnet_exact, diffusion=NU_TRUE)

print("left weight:")
print(symnet_exact.left.weight.detach().cpu().numpy())
print("right weight:")
print(symnet_exact.right.weight.detach().cpu().numpy())
print("linear weight:")
print(symnet_exact.linear.weight.detach().cpu().numpy())
print("product readout weight:")
print(symnet_exact.product_readout.weight.detach().cpu().numpy())

exact_coeffs = product_term_dict(symnet_exact)
print()
print("Expanded polynomial:")
print(pretty_equation(exact_coeffs))
print()
print("By inspection:")
print("left(features)  = u")
print("right(features) = u_x")
print("product_readout(left * right) = -1 * (u * u_x)")
print("linear(features) = 0.02 * u_xx")
print("total = -u*u_x + 0.02*u_xx")

reference_features = np.column_stack([
    reference["u"]["physical"].reshape(-1),
    reference["u_x"]["physical"].reshape(-1),
    reference["u_xx"]["physical"].reshape(-1),
]).astype(np.float32)

reference_rhs = (-reference["u"]["physical"] * reference["u_x"]["physical"] + NU_TRUE * reference["u_xx"]["physical"]).reshape(-1, 1)

with torch.no_grad():
    symnet_exact_rhs = symnet_exact(torch.tensor(reference_features, dtype=torch.float32, device=device)).cpu().numpy()

symnet_exact_metrics = pd.DataFrame([
    as_metric_row("hand-set SymNet vs exact Burgers RHS", symnet_exact_rhs.reshape(u_grid.shape), reference_rhs.reshape(u_grid.shape)),
]).set_index("quantity")
display(symnet_exact_metrics)

## 7. Train SymNet Starting from the Exact Burgers Initialization

Question answered here: once the SIREN is frozen, if SymNet starts exactly at the Burgers formula and is trained only to match the surrogate's autodiff `u_t`, does it remain near Burgers or drift toward a different lower-loss representation?

In [ ]:
frozen_feature_tensor = torch.tensor(
    np.column_stack([
        surrogate_fields["u"].reshape(-1),
        surrogate_fields["u_x"].reshape(-1),
        surrogate_fields["u_xx"].reshape(-1),
    ]).astype(np.float32),
    dtype=torch.float32,
    device=device,
)
frozen_ut_target = torch.tensor(
    surrogate_fields["u_t"].reshape(-1, 1).astype(np.float32),
    dtype=torch.float32,
    device=device,
)

symnet_frozen = MinimalSymNet().to(device)
initialize_symnet_to_burgers(symnet_frozen, diffusion=NU_TRUE)

symnet_steps = 400
symnet_lr = 1e-2
symnet_log_every = 40
symnet_optimizer = torch.optim.Adam(symnet_frozen.parameters(), lr=symnet_lr)
symnet_loss_fn = nn.MSELoss()

symnet_history = []

for step in range(symnet_steps + 1):
    pred_ut = symnet_frozen(frozen_feature_tensor)
    pde_loss = symnet_loss_fn(pred_ut, frozen_ut_target)

    coeffs = product_term_dict(symnet_frozen)
    row = {
        "step": step,
        "pde_loss": float(pde_loss.item()),
        "u": coeffs["u"],
        "u_x": coeffs["u_x"],
        "u_xx": coeffs["u_xx"],
        "u^2": coeffs["u^2"],
        "u*u_x": coeffs["u*u_x"],
        "u*u_xx": coeffs["u*u_xx"],
        "u_x^2": coeffs["u_x^2"],
        "u_x*u_xx": coeffs["u_x*u_xx"],
        "u_xx^2": coeffs["u_xx^2"],
    }
    symnet_history.append(row)

    if step % symnet_log_every == 0:
        print(
            f"step {step:4d} | pde_loss = {row['pde_loss']:.6e} | "
            f"c(u*u_x) = {row['u*u_x']:+.6f} | c(u_xx) = {row['u_xx']:+.6f} | "
            f"unwanted u^2 = {row['u^2']:+.6f} | unwanted u_x^2 = {row['u_x^2']:+.6f}"
        )

    if step == symnet_steps:
        break

    symnet_optimizer.zero_grad()
    pde_loss.backward()
    symnet_optimizer.step()

symnet_history_df = pd.DataFrame(symnet_history)
display(symnet_history_df.head())
print()
print("final expanded equation:")
print(pretty_equation(product_term_dict(symnet_frozen)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(symnet_history_df["step"], symnet_history_df["pde_loss"], linewidth=2)
axes[0].set_title("Frozen-surrogate SymNet PDE loss")
axes[0].set_xlabel("step")
axes[0].set_ylabel("MSE")
axes[0].set_yscale("log")

axes[1].plot(symnet_history_df["step"], symnet_history_df["u*u_x"], label="u*u_x", linewidth=2)
axes[1].plot(symnet_history_df["step"], symnet_history_df["u_xx"], label="u_xx", linewidth=2)
axes[1].plot(symnet_history_df["step"], symnet_history_df["u^2"], label="u^2", alpha=0.8)
axes[1].plot(symnet_history_df["step"], symnet_history_df["u_x^2"], label="u_x^2", alpha=0.8)
axes[1].plot(symnet_history_df["step"], symnet_history_df["u*u_xx"], label="u*u_xx", alpha=0.8)
axes[1].axhline(-1.0, color="black", linestyle="--", linewidth=1)
axes[1].axhline(0.02, color="gray", linestyle=":", linewidth=1)
axes[1].set_title("Coefficient trajectories")
axes[1].set_xlabel("step")
axes[1].set_ylabel("effective coefficient")
axes[1].legend(loc="best")

plt.tight_layout()
plt.show()

display(symnet_history_df.tail())

## 8. Optional Next Experiment: Joint SIREN + SymNet Training

Question answered here: if we start from the same exact Burgers SymNet initialization but now let both the surrogate and SymNet move, how do the data loss, PDE loss, derivative fidelity, Burgers coefficients, and weight drift compare against the frozen-surrogate case above?

This section is intentionally off by default so it does not obscure the simpler experiments.

In [ ]:
RUN_OPTIONAL_JOINT = False
joint_steps = 300
joint_batch_size = 4096
joint_lr = 5e-4
joint_lambda_pde = 1.0
joint_log_every = 50

if RUN_OPTIONAL_JOINT:
    joint_siren = copy.deepcopy(siren).to(device)
    joint_symnet = MinimalSymNet().to(device)
    initialize_symnet_to_burgers(joint_symnet, diffusion=NU_TRUE)

    joint_optimizer = torch.optim.Adam(
        list(joint_siren.parameters()) + list(joint_symnet.parameters()),
        lr=joint_lr,
    )
    joint_history = []

    for step in range(joint_steps + 1):
        idx = torch.randint(0, len(t_train), (joint_batch_size,), device=device)
        t_batch = t_train[idx].clone().detach().requires_grad_(True)
        x_batch = x_train[idx].clone().detach().requires_grad_(True)
        u_batch_true = u_train[idx]

        u_batch_pred = joint_siren(t_batch, x_batch)
        ut_batch = torch.autograd.grad(
            u_batch_pred,
            t_batch,
            grad_outputs=torch.ones_like(u_batch_pred),
            create_graph=True,
            retain_graph=True,
        )[0]
        ux_batch = torch.autograd.grad(
            u_batch_pred,
            x_batch,
            grad_outputs=torch.ones_like(u_batch_pred),
            create_graph=True,
            retain_graph=True,
        )[0]
        uxx_batch = torch.autograd.grad(
            ux_batch,
            x_batch,
            grad_outputs=torch.ones_like(ux_batch),
            create_graph=True,
            retain_graph=True,
        )[0]

        feat_batch = torch.cat([u_batch_pred, ux_batch, uxx_batch], dim=1)
        rhs_batch = joint_symnet(feat_batch)

        data_loss = loss_fn(u_batch_pred, u_batch_true)
        pde_loss = loss_fn(ut_batch, rhs_batch)
        total_loss = data_loss + joint_lambda_pde * pde_loss

        if step % joint_log_every == 0:
            full_fields = evaluate_model_and_derivatives_in_chunks(
                joint_siren,
                t_flat=t_flat,
                x_flat=x_flat,
                grid_shape=u_grid.shape,
                chunk_size=4096,
                device=device,
            )
            coeffs = product_term_dict(joint_symnet)
            joint_history.append({
                "step": step,
                "data_loss": float(data_loss.item()),
                "pde_loss": float(pde_loss.item()),
                "u_rel_l2": compute_error_metrics(full_fields["u"], u_grid)["rel_l2"],
                "ut_rel_l2": compute_error_metrics(full_fields["u_t"], reference["u_t"]["physical"])["rel_l2"],
                "ux_rel_l2": compute_error_metrics(full_fields["u_x"], reference["u_x"]["physical"])["rel_l2"],
                "uxx_rel_l2": compute_error_metrics(full_fields["u_xx"], reference["u_xx"]["physical"])["rel_l2"],
                "u*u_x": coeffs["u*u_x"],
                "u_xx": coeffs["u_xx"],
                "u^2": coeffs["u^2"],
                "u_x^2": coeffs["u_x^2"],
            })
            print(
                f"joint step {step:4d} | data_loss = {float(data_loss.item()):.6e} | "
                f"pde_loss = {float(pde_loss.item()):.6e} | c(u*u_x) = {coeffs['u*u_x']:+.6f} | c(u_xx) = {coeffs['u_xx']:+.6f}"
            )

        if step == joint_steps:
            break

        joint_optimizer.zero_grad()
        total_loss.backward()
        joint_optimizer.step()

    joint_history_df = pd.DataFrame(joint_history)
    display(joint_history_df)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(joint_history_df["step"], joint_history_df["data_loss"], label="data_loss")
    axes[0].plot(joint_history_df["step"], joint_history_df["pde_loss"], label="pde_loss")
    axes[0].set_yscale("log")
    axes[0].set_title("Joint losses")
    axes[0].legend(loc="best")

    axes[1].plot(joint_history_df["step"], joint_history_df["u*u_x"], label="u*u_x")
    axes[1].plot(joint_history_df["step"], joint_history_df["u_xx"], label="u_xx")
    axes[1].plot(joint_history_df["step"], joint_history_df["u^2"], label="u^2")
    axes[1].plot(joint_history_df["step"], joint_history_df["u_x^2"], label="u_x^2")
    axes[1].set_title("Joint coefficient drift")
    axes[1].legend(loc="best")

    axes[2].plot(joint_history_df["step"], joint_history_df["u_rel_l2"], label="u")
    axes[2].plot(joint_history_df["step"], joint_history_df["ut_rel_l2"], label="u_t")
    axes[2].plot(joint_history_df["step"], joint_history_df["ux_rel_l2"], label="u_x")
    axes[2].plot(joint_history_df["step"], joint_history_df["uxx_rel_l2"], label="u_xx")
    axes[2].set_title("Derivative fidelity during joint training")
    axes[2].legend(loc="best")

    plt.tight_layout()
    plt.show()
else:
    print("Optional joint experiment skipped. Set RUN_OPTIONAL_JOINT = True to execute it.")